# TinyDoc-VLM: Colab Training

Trains all 3 stages on T4 (free tier). Auto-resumes and auto-advances through stages.

**IMPORTANT**:
- Runtime → Change runtime type → **T4 GPU**
- Run cells in order. If disconnected, re-run from the top — auto-resume picks up where you left off.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/tinydoc-vlm'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
print(f'Drive ready at {DRIVE_ROOT}')

## 2. Clone repo & install deps

In [ ]:
import os, sys
REPO_DIR = '/content/tinydoc-vlm'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/eulogik/TinyDoc-VLM.git {REPO_DIR}

%cd {REPO_DIR}
!git pull
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers sentencepiece tokenizers pillow numpy pandas tqdm pyyaml gradio optimum onnx onnxruntime einops faker jinja2 pydantic
!pip install -q accelerate
print('Deps installed')

## 3. Prepare data

Copies from Drive if already generated, otherwise generates 10K synthetic docs.

In [ ]:
import shutil
from pathlib import Path

DATA_DIR = 'data/synthetic'
DRIVE_DATA = f'{DRIVE_ROOT}/data/synthetic'
MANIFEST = Path(DATA_DIR) / 'output' / 'manifest.jsonl'

if MANIFEST.exists():
    print(f'Data already exists locally ({sum(1 for _ in open(MANIFEST))} docs) — skipping copy')
elif os.path.exists(f'{DRIVE_DATA}/output/manifest.jsonl'):
    print('Copying data from Drive...')
    shutil.copytree(DRIVE_DATA, DATA_DIR, dirs_exist_ok=True)
    print(f'Done. Manifest: {sum(1 for _ in open(MANIFEST))} docs')
else:
    print('Generating 10K synthetic documents (~15 min)...')
    !python data/synthetic/generator.py --num-docs 10000 --output-dir data/synthetic/output
    print('Copying data to Drive for future sessions...')
    shutil.copytree(DATA_DIR, DRIVE_DATA, dirs_exist_ok=True)
assert MANIFEST.exists(), "Generation failed — check output above"

## 4. Train all stages (auto-detect & auto-advance)

Runs Stage 1 → 2 → 3 sequentially. Each stage picks up from its Drive checkpoint. Completed stages are skipped.

In [ ]:
import yaml
import torch
import logging
from torch.utils.data import random_split
from pathlib import Path

from tinydoc_vlm import (
    TinyDocVLMConfig,
    TinyDocVLMForConditionalGeneration,
    TinyDocVLMProcessor,
    TinyDocVLMTrainer,
    TrainerConfig,
    DocumentDataset,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('colab')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f'Device: {device}')

STAGE_CONFIGS = {
    1: 'training/stage1_layout_pretrain.yaml',
    2: 'training/stage2_doc_understanding.yaml',
    3: 'training/stage3_instruction_tuning.yaml',
}

def build_config(stage):
    with open(STAGE_CONFIGS[stage]) as f:
        cfg = yaml.safe_load(f)
    cfg['training']['batch_size'] = 4
    cfg['training']['gradient_accumulation_steps'] = 8
    cfg['training']['save_every_steps'] = 999999
    cfg['training']['eval_every_steps'] = 250
    cfg['training']['log_every_steps'] = 10
    cfg['training']['gradient_checkpointing'] = True
    cfg['training']['num_epochs'] = 1
    if stage > 1:
        cfg['model']['pretrained_checkpoint'] = f'{DRIVE_ROOT}/checkpoints/stage{stage-1}/best'
    cfg['training']['output_dir'] = f'{DRIVE_ROOT}/checkpoints/stage{stage}'
    cfg['data']['manifest_path'] = 'data/synthetic/output/manifest.jsonl'
    cfg['training']['learning_rate'] = float(cfg['training']['learning_rate'])
    return cfg

def run_stage(stage):
    logger.info(f'=== Stage {stage} ===')
    cfg = build_config(stage)
    tc = cfg['training']
    checkpoint_dir = Path(tc['output_dir'])
    stage_file = checkpoint_dir / 'stage_complete.txt'

    if stage_file.exists():
        logger.info(f'Stage {stage} already complete. Skipping.')
        return

    mc = cfg['model']
    config = TinyDocVLMConfig(
        vision_config=mc.get('vision_config'),
        decoder_config=mc.get('decoder_config'),
        pixel_shuffle_scale=mc.get('pixel_shuffle_scale', 3),
        image_size=mc.get('image_size', 384),
        patch_size=mc.get('patch_size', 16),
    )
    model = TinyDocVLMForConditionalGeneration(config)

    pretrained = mc.get('pretrained_checkpoint')
    if pretrained and os.path.exists(pretrained):
        logger.info(f'Loading pretrained from: {pretrained}')
        model = TinyDocVLMForConditionalGeneration.from_pretrained(pretrained)

    dc = cfg['data']
    dataset = DocumentDataset(
        data_root=dc.get('data_root', 'data/synthetic'),
        manifest_path=dc.get('manifest_path'),
        max_seq_length=dc.get('max_seq_length', 2048),
        stage=stage,
    )

    train_size = int(0.9 * len(dataset))
    eval_size = len(dataset) - train_size
    train_dataset, eval_dataset = random_split(dataset, [train_size, eval_size])
    logger.info(f'Dataset: {len(dataset)} total ({train_size} train / {eval_size} eval)')

    trainer_cfg = TrainerConfig(
        output_dir=str(checkpoint_dir),
        num_epochs=tc.get('num_epochs', 1),
        batch_size=tc.get('batch_size', 4),
        gradient_accumulation_steps=tc.get('gradient_accumulation_steps', 8),
        learning_rate=float(tc.get('learning_rate', 1e-4)),
        min_learning_rate=float(tc.get('min_learning_rate', 1e-5)),
        warmup_steps=100,
        weight_decay=tc.get('weight_decay', 0.01),
        max_grad_norm=tc.get('max_grad_norm', 1.0),
        max_seq_length=tc.get('max_seq_length', 2048),
        stage=stage,
        use_fp16=tc.get('use_fp16', True),
        save_every_steps=999999,
        eval_every_steps=tc.get('eval_every_steps', 250),
        log_every_steps=tc.get('log_every_steps', 10),
        gradient_checkpointing=tc.get('gradient_checkpointing', True),
    )

    processor = TinyDocVLMProcessor()
    trainer = TinyDocVLMTrainer(
        model=model, processor=processor,
        train_dataset=train_dataset, eval_dataset=eval_dataset,
        config=trainer_cfg, device=device,
    )

    if (checkpoint_dir / 'trainer_state.pt').exists():
        ckpts = sorted(checkpoint_dir.glob('epoch_*'))
        if ckpts:
            trainer.load_checkpoint(str(ckpts[-1]))
            logger.info(f'Resumed from {ckpts[-1]}')

    trainer.train()

    stage_file.write_text('done')
    for d in checkpoint_dir.glob('init'):
        shutil.rmtree(d)
    for d in sorted(checkpoint_dir.glob('step_*'))[:-1]:
        shutil.rmtree(d)
    logger.info(f'Stage {stage} complete.')

# --- Run all 3 stages ---
for s in [1, 2, 3]:
    run_stage(s)

logger.info('All 3 stages complete! Move to cell 5 to download the model.')

## 5. Download final model (optional)

Downloads `stage3/best` after all 3 stages complete.

In [ ]:
from google.colab import drive, files
import os

DRIVE_ROOT = '/content/drive/MyDrive/tinydoc-vlm'

final_dir = f'{DRIVE_ROOT}/checkpoints/stage3/best'
if os.path.exists(final_dir):
    !zip -r /content/tinydoc-vlm-final.zip {final_dir}
    files.download('/content/tinydoc-vlm-final.zip')
    print('Download started!')
else:
    print('Stage 3 not yet complete. Run cell 4 first.')